# Unified Phase Retrieval Core

This notebook demonstrates `library/phase_retrieval_core_unified.py`: ordinary two-helicity use, arbitrary labeled hologram dictionaries, and the `Nmodes` switch between the fast single-mode kernel and the multimode kernel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from library import phase_retrieval_core_unified as pr

## Synthetic Inputs

Replace these arrays with centered measured hologram intensities. All holograms in the dictionary must have the same `(nx, ny)` shape.

In [ ]:
rng = np.random.default_rng(3)
shape = (64, 64)
yy, xx = np.indices(shape)
rr = np.hypot(xx - shape[1] / 2, yy - shape[0] / 2)

base = 2.0 + np.exp(-(rr / 12) ** 2)
pos = base + 0.15 * rng.random(shape)
neg = 0.92 * base + 0.15 * rng.random(shape)
LH = 1.08 * base + 0.15 * rng.random(shape)
LV = 0.75 * base + 0.15 * rng.random(shape)

holograms = {"pos": pos, "neg": neg, "LH": LH, "LV": LV}
mask_pixel = np.zeros(shape, dtype=int)
mask_pixel[30:34, 30:34] = 1
supportmask = (rr < 18).astype(float)

## Labeled Hologram Sequence

The recipe key `helicity` is kept for compatibility with the old recipe format. In this module it means the hologram dictionary key used at each step.

In [ ]:
recipe = {
    "algorithm_list": ["ER", "ER", "ER", "ER"],
    "number_iterations": [5, 5, 5, 5],
    "helicity": ["pos", "neg", "LH", "LV"],
    "beta_zero": [0.5, 0.5, 0.5, 0.5],
    "beta_mode": ["const", "const", "const", "const"],
    "alpha_zero": [0.0, 0.0, 0.0, 0.0],
    "alpha_mode": ["const", "const", "const", "const"],
    "RL_its": [0, 0, 0, 0],
    "RL_freqs": [1e9, 1e9, 1e9, 1e9],
    "TV_freqs": [1e9, 1e9, 1e9, 1e9],
    "plot_every": [2, 2, 2, 2],
    "average_img": [2, 2, 2, 2],
    "Fourier_last": [True, True, True, True],
    "Startimage": [None, "pos", "neg", "LH"],
    "Startgamma": [None, None, None, None],
    "Nmodes": 1,
    "normalize_startimage_between_holograms": True,
}

result = pr.phase_retrieval_algorithm(holograms, mask_pixel, supportmask, recipe)
result["full_coherence"].keys()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, label in zip(axes, recipe["helicity"]):
    field = result["full_coherence"][label]
    ax.imshow(np.log10(np.abs(field) + 1e-9), cmap="magma")
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout()

## Multimode Switch

`Nmodes == 1` uses the fast single-mode core. `Nmodes > 1` uses the multimode summed-intensity constraint.

In [ ]:
multi_recipe = recipe | {
    "Nmodes": 2,
    "Startimage": [
        np.stack([np.sqrt(pos), 1j * np.sqrt(pos)]) / np.sqrt(2),
        "pos",
        "neg",
        "LH",
    ],
}
multi_result = pr.phase_retrieval_algorithm(holograms, mask_pixel, supportmask, multi_recipe)
multi_result["full_coherence"]["LV"].shape